# 06. CNN 体系

卷积神经网络（CNN）解决了 MLP 在图像任务上的两个关键问题：参数太多、没有利用局部空间结构。

## 卷积神经网络（CNN）的形式化定义

卷积神经网络是一类面向网格结构数据的深度模型，其核心操作是局部感受野上的卷积变换与跨空间位置的参数共享。该结构能够有效提取局部空间模式，并以较低参数代价构造层级特征表示。

## 输入、输出与参数化方式

            输入通常是二维或三维张量，例如图像张量 $X \in \mathbb{R}^{C 	imes H 	imes W}$。卷积层参数以卷积核形式存在，其输出是新的特征图张量。

经过多层卷积和下采样之后，网络把低层局部纹理逐步变换为高层语义特征，最终由分类头、检测头或分割头完成任务输出。

## 结构分解与信息流

            CNN 的结构可分为特征提取主干与任务头两部分：

- 特征提取主干：卷积、激活、归一化、池化
- 任务头：全连接层或其他预测模块

参数共享与局部连接是 CNN 的关键归纳偏置。前者显著降低参数量，后者使网络天然适应空间邻域模式。

## 优化目标与训练机制

            CNN 的训练方式与其他深度网络一致，但其表示学习行为具有明显层次性。浅层卷积核通常对边缘、纹理等局部低级模式敏感，深层卷积块则更关注任务相关的高层语义模式。

在训练中，卷积核、归一化参数和分类头都会通过反向传播共同优化。

## 计算复杂度、统计性质与工程代价

            CNN 在视觉任务中的优势来自较好的参数效率和空间归纳偏置。相比全连接网络，它的参数量通常显著更低；相比纯注意力结构，它在中小视觉任务上常常更具计算性价比。

其局限在于：长距离依赖需要依靠更深网络或更大感受野间接建模。

## 与相邻模型的差异

            与 MLP 相比，CNN 显式利用空间结构。
与 Vision Transformer 相比，CNN 具有更强的局部先验，但全局建模通常不如注意力直接。
在资源受限或样本量不大的视觉任务中，CNN 仍然具有很强的现实优势。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(13, 5))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 5)

stages = [
    (1.2, 2.5, 1.2, 1.8, "#4C78A8", "输入图像\nH x W"),
    (3.5, 2.5, 1.4, 1.8, "#F58518", "卷积 + ReLU\n局部特征"),
    (5.9, 2.5, 1.4, 1.6, "#E45756", "池化\n降采样"),
    (8.2, 2.5, 1.4, 1.8, "#72B7B2", "卷积块\n高阶特征"),
    (10.4, 2.5, 1.2, 1.6, "#54A24B", "Flatten"),
    (12.0, 2.5, 1.1, 1.5, "#B279A2", "分类头"),
]
for x, y, w, h, color, text in stages:
    rect = plt.Rectangle((x - w/2, y - h/2), w, h, facecolor=color, edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=11)
for i in range(len(stages) - 1):
    ax.annotate("", xy=(stages[i+1][0] - stages[i+1][2]/2, 2.5), xytext=(stages[i][0] + stages[i][2]/2, 2.5),
                arrowprops=dict(arrowstyle="->", lw=1.8))
ax.set_title("卷积神经网络的数据流")
plt.show()

## 先建立直觉

            CNN 的核心直觉很简单：

图像不是普通向量，而是有空间结构的网格。

当你看到一张数字图片时，你不会把 64 个像素点当成 64 个彼此独立的数字；
你会注意局部笔画、边缘、拐角，再逐步组合成完整数字。

CNN 正是在做这件事：

- 先识别局部模式
- 再把局部模式组合成更高级特征

## 架构是怎么工作的

            CNN 的关键组件有三个：

- 卷积层：用共享卷积核扫过图像，提取局部特征
- 激活函数：让表示变得非线性
- 池化层：压缩空间尺寸，增强鲁棒性

前层卷积核常学到边缘、方向、局部纹理；
后层会逐渐学到更抽象的笔画组合或局部形状。

这就是为什么 CNN 对图像比 MLP 更自然。

## 训练时到底发生了什么

            训练 CNN 和训练 MLP 的流程本质上完全一致，
不同点只在于前向传播时使用了卷积和池化。

真正需要理解的是：卷积核不是手工写好的，它们也是参数，会通过反向传播自动学出来。

所以 CNN 的强大之处，不是“卷积这个运算本身很神奇”，
而是“它给模型施加了更适合图像的结构归纳偏置”。

## 什么时候该用它

            CNN 适合：

- 图像分类
- 图像检测与分割的主干特征提取
- 局部模式很重要的二维或一维信号任务

即使在 Transformer 很流行之后，CNN 仍然是视觉任务里非常重要的基础。

## 最常见的误区

            常见误区：

1. **误以为卷积核是在做固定滤波**
   在深度学习里，卷积核是可学习参数，不是手工滤波器。

2. **误以为池化一定越多越好**
   池化太强会丢失细节。

3. **误以为 CNN 已经过时**
   它在很多中小视觉任务、边缘设备、轻量模型中依然非常强。

## 1. 卷积的公式

$$
Y(i, j) = \sum_m \sum_n X(i+m, j+n) K(m, n)
$$

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)
X = X[:, None, :, :]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=256, shuffle=False)

In [ ]:
class LinearImageBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.net(x)


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 2 * 2, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.classifier(self.features(x))


class CNNWithBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, train_loader, test_loader, epochs=20, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"test_loss": [], "test_acc": []}

    for _ in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

        model.eval()
        test_loss, test_correct, test_total = 0.0, 0, 0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                test_loss += loss.item() * batch_x.size(0)
                test_correct += (logits.argmax(dim=1) == batch_y).sum().item()
                test_total += batch_x.size(0)

        history["test_loss"].append(test_loss / test_total)
        history["test_acc"].append(test_correct / test_total)
    return history

In [ ]:
cnn_models = {
    "LinearBaseline": LinearImageBaseline(),
    "SimpleCNN": SimpleCNN(),
    "CNN+BatchNorm": CNNWithBatchNorm(),
}

cnn_histories = {}
trained_cnn_models = {}
for name, model in cnn_models.items():
    history = train_model(model, train_loader, test_loader, epochs=20, lr=1e-3)
    cnn_histories[name] = history
    trained_cnn_models[name] = model
    print(name, "最终测试准确率:", round(history["test_acc"][-1], 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for name, history in cnn_histories.items():
    axes[0].plot(history["test_loss"], label=name)
    axes[1].plot(history["test_acc"], label=name)
axes[0].set_title("CNN 测试损失曲线")
axes[1].set_title("CNN 测试准确率曲线")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
best_name = max(cnn_histories, key=lambda name: cnn_histories[name]["test_acc"][-1])
best_model = trained_cnn_models[best_name]

best_model.eval()
with torch.no_grad():
    logits = best_model(torch.tensor(X_test))
    preds = logits.argmax(dim=1).numpy()

print("最佳模型:", best_name)
print("测试准确率:", round(accuracy_score(y_test, preds), 4))

plt.figure(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, preds, cmap="Blues")
plt.title(f"{best_name} 的混淆矩阵")
plt.show()